# Preliminary data processing

This script takes in the results from assembles-datasets and creates certain userful variables in the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
from pydmd import DMD
import os
import plotly.graph_objects as go
import plotly.express as px
import pickle

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns
from celluloid import Camera

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [2]:
# Load data from infections
reg_model = ''
runs = '-1-'
comment = "sparse-reg" #"Nact-Ediv-vir" #"prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg # comp_bias-Nact-Ediv-vir

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Nact_I,psi_Nact_H,psi_Nact_P,F0_Nact,psi_NM_I,psi_NM_H,psi_NM_P,F0_NM,psi_EM_I,psi_EM_H,psi_EM_P,F0_EM,psi_Ediv_I,psi_Ediv_H,psi_Ediv_P,F0_Ediv,d_I,K_IE,b_I,K_EH,S_0,I_0,d_S,d_IE,b_H,d_H,N_0,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_act,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_pM,T_pM_min,int_pE,int_pH,min_pS
0,-2.0,-2.0,-2.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.00,1.000000e+07,0.000000e+00,50000.0,10000000.0,0.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000,300.0,0.000000,20.000000,0.0,0.000000,0.000000,0.000000e+00,1.000000e+07
1,-2.0,-2.0,-2.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.00,1.000000e+08,0.000000e+00,50000.0,10000000.0,0.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000,300.0,0.000000,20.000000,0.0,0.000000,0.000000,0.000000e+00,1.000000e+07
2,-2.0,-2.0,-2.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+07,0.000000e+00,50000.0,10000000.0,0.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000,300.0,0.000000,20.000000,0.0,0.000000,0.000000,4.440145e+04,9.912799e+06
3,-2.0,-2.0,-2.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+08,0.000000e+00,50000.0,10000000.0,0.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000,300.0,0.000000,20.000000,0.0,0.000000,0.000000,4.440145e+04,9.912799e+06
4,-2.0,-2.0,-2.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.25,1.000000e+04,2.000000e-07,5000.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,4.190458e+07,6.548571,0.0,1.047475e+07,0.000000,300.0,0.000000,20.000000,0.0,0.000000,0.000000,5.205050e+06,1.521530e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94429995,2.0,2.0,2.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.25,5.994843e+05,2.000000e-07,50000.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,5.221209e+06,11.257143,0.0,6.550901e+06,5784.045781,12932.0,8.388571,7.108190,881455.0,39.745329,4557.405714,3.262757e+06,3.945499e+06
94429996,2.0,2.0,2.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.25,5.994843e+05,2.000000e-07,500000.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,5.212952e+06,11.200000,0.0,6.545753e+06,5213.928111,13198.0,10.982857,8.322476,1685149.0,43.017911,4617.714286,3.260440e+06,3.951805e+06
94429997,2.0,2.0,2.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.25,1.000000e+06,2.000000e-07,5000.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,5.250265e+06,11.245714,0.0,6.572535e+06,5442.570643,10668.0,6.891429,5.828229,422356.0,36.017321,4009.257143,3.273926e+06,3.925207e+06
94429998,2.0,2.0,2.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.25,1.000000e+06,2.000000e-07,50000.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,300.0,4.0,4.0,4.0,1.0,0.75,0.75,0.25,0.333333,0.5,2.5,0.25,5.237210e+06,11.211429,0.0,6.563539e+06,5188.961269,11554.0,8.582857,7.384495,602884.0,37.823425,4150.434286,3.269446e+06,3.934920e+06


## 1. Understanding the statistics of responses to an infection

In [3]:
# Create additional variables
virs = np.unique(mean_df[['I_0','d_I','K_IE','b_I','N_0','K_EH']].to_numpy(), axis = 0)

module_reg = {'Nact': Nact_reg, 'NM': NM_reg, 'EM': EM_reg, 'Ediv': Ediv_reg}
module_reg_labels = {'Nact': param_names[-16:-12], 'NM': param_names[-12:-8], 'EM': param_names[-8:-4], 'Ediv': param_names[-4:]}

if "full" in comment:
    reg = Nact_reg + NM_reg + EM_reg + Ediv_reg
    reg_label = param_names[-len(reg):]
    modules = ['Nact', 'NM', 'EM', 'Ediv']
    reg_and = [Nact_reg[2]] + [NM_reg[2]] + [EM_reg[2]] + [Ediv_reg[2]]
    reg_bias = [Nact_reg[3]] + [NM_reg[3]] + [EM_reg[3]] + [Ediv_reg[3]]
else:
    reg = (Nact_reg if "Nact" in comment else []) + (NM_reg if "NM" in comment else []) + (EM_reg if "EM" in comment else []) + (Ediv_reg if "Ediv" in comment else [])
    reg_label = (param_names[-16:-12] if "Nact" in comment else []) + (param_names[-12:-8] if "NM" in comment else []) + (param_names[-8:-4] if "EM" in comment else []) + (param_names[-4:] if "Ediv" in comment else [])
    modules = (["Nact"] if "Nact" in comment else []) + (["NM"] if "NM" in comment else []) + (["EM"] if "EM" in comment else []) + (["Ediv"] if "Ediv" in comment else [])
    reg_or = ([Nact_reg[0:2]] if "Nact" in comment else []) + ([NM_reg[0:2]] if "NM" in comment else []) + ([EM_reg[0:2]] if "EM" in comment else []) + ([Ediv_reg[0:2]] if "Ediv" in comment else [])
    reg_and = ([Nact_reg[2]] if "Nact" in comment else []) + ([NM_reg[2]] if "NM" in comment else []) + ([EM_reg[2]] if "EM" in comment else []) + ([Ediv_reg[2]] if "Ediv" in comment else [])
    reg_and_label = ([param_names[-14]] if "Nact" in comment else []) + ([param_names[-10]] if "NM" in comment else []) + ([param_names[-6]] if "EM" in comment else []) + ([param_names[-2]] if "Ediv" in comment else [])
    reg_bias = ([Nact_reg[3]] if "Nact" in comment else []) + ([NM_reg[3]] if "NM" in comment else []) + ([EM_reg[3]] if "EM" in comment else []) + ([Ediv_reg[3]] if "Ediv" in comment else [])
    reg_bias_label = ([param_names[-13]] if "Nact" in comment else []) + ([param_names[-9]] if "NM" in comment else []) + ([param_names[-5]] if "EM" in comment else []) + ([param_names[-1]] if "Ediv" in comment else [])

# Build variables

mean_df['antigenicity_over_harm'] = antigenicity_over_harm(mean_df)
mean_df['T_pE_clear'] = mean_df['T_pE_max'] - mean_df['T_max_pI']
mean_df['max_pM_fold'] = mean_df['max_pM']/mean_df['N_0']

mean_df['max_pE_fold'] = mean_df['max_pE']/mean_df['N_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI','T_min_pI', 'harm_pS', "max_pM_fold", "T_pM_min", "T_max_pI", "T_pE_start", "T_pE_max", "max_pE_fold", 'min_pS', 'antigenicity_over_harm', 'T_pE_clear'] 
#keep_vars = ['harm_pI','T_min_pI', 'harm_pS', "max_pM_fold", "T_max_pI", "T_pE_start", "T_pE_max", "int_pE_fold", 'min_pS', 'antigenicity_over_harm', 'T_pE_clear']

In [4]:
# save data sets
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_IE, b_I, N_0, K_EH) in enumerate(tqdm(virs)):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I)*(mean_df["K_EH"] == K_EH), ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'K_EH'] + Nact_reg + NM_reg + EM_reg + Ediv_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_IE = K_IE, d_I = d_I, b_I = b_I) #, vir_model = "indep_harm" if b_I > 0 and d_I > 10*d_S else "autoimmune")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/(b_S*sim_duration)
    data.loc[:,"peff_protection"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"peff_utility"] = (no_eff_stats[3] + 1.0*no_eff_stats[4] - (data['harm_pI'] + data['harm_pS']).to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)
clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 304/304 [27:27<00:00,  5.42s/it]


In [5]:
del clustered_mean_df, infection_scenarios